# Notebook básico: DVC + MLflow

Exemplo mínimo para deixar no GitHub e entender o fluxo:

- gerar um dataset simples
- treinar um modelo
- logar parâmetros e métricas no **MLflow**
- salvar artefatos
- mostrar como o **DVC** entraria no versionamento do dataset/modelo

> Esse notebook foi pensado para ser **didático e pequeno**, não para produção.

## 1) Instalação

Se estiver rodando localmente, descomente a célula abaixo.

In [1]:
# %pip install mlflow scikit-learn pandas matplotlib joblib

  Using cached pandas-2.3.3-cp311-cp311-macosx_11_0_arm64.whl.metadata (91 kB)
  Using cached aiohappyeyeballs-2.6.1-py3-none-any.whl.metadata (5.9 kB)
  Using cached aiosignal-1.4.0-py3-none-any.whl.metadata (3.7 kB)
  Using cached frozenlist-1.8.0-cp311-cp311-macosx_11_0_arm64.whl.metadata (20 kB)
  Using cached propcache-0.4.1-cp311-cp311-macosx_11_0_arm64.whl.metadata (13 kB)
  Using cached pyasn1_modules-0.4.2-py3-none-any.whl.metadata (3.5 kB)
  Using cached zipp-3.23.0-py3-none-any.whl.metadata (3.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 19.4 MB/s  0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 25.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 25.4 MB/s  0:00:00
Using cached pandas-2.3.3-cp311-cp311-macosx_11_0_arm64.whl (10.8 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 25.2 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 838.5/838.5 kB 31.0 MB/s  0:00:00
   ━━━━━━━━━━━━

## 2) Imports

In [2]:
import os
from pathlib import Path

import joblib
import mlflow
import mlflow.sklearn
import pandas as pd

from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split

/Users/yandrade/PycharmProjects/ia_study/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 3) Estrutura de pastas

Vamos criar uma estrutura mínima para o exemplo:

In [3]:
Path("data").mkdir(exist_ok=True)
Path("models").mkdir(exist_ok=True)
Path("artifacts").mkdir(exist_ok=True)

## 4) Criando um dataset sintético

Aqui a ideia é só ter um arquivo `.csv` real para o DVC poder versionar.

In [4]:
X, y = make_classification(
    n_samples=500,
    n_features=8,
    n_informative=5,
    n_redundant=1,
    random_state=42
)

feature_names = [f"feature_{i}" for i in range(X.shape[1])]
df = pd.DataFrame(X, columns=feature_names)
df["target"] = y

dataset_path = "data/sample_dataset.csv"
df.to_csv(dataset_path, index=False)

df.head()

,feature_0,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,target
0,0.682069,0.539653,2.779035,-0.599449,0.263499,0.050225,0.126178,0.540071,1
1,-1.130615,-0.628521,1.897531,0.914287,0.058928,-0.519921,0.622711,1.496569,0
2,-1.466785,-0.776843,-1.588155,0.785985,-2.824823,1.112359,-0.501402,-1.293043,0
3,0.458168,-2.569133,0.587566,-1.369820,-0.339987,-0.340855,-0.979721,-1.384609,0
4,1.476934,-0.699262,0.752082,1.060880,1.123465,3.998698,-1.167780,-1.697894,0


## 5) Separando treino e teste

In [5]:
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["target"]
)

X_train = train_df.drop(columns="target")
y_train = train_df["target"]

X_test = test_df.drop(columns="target")
y_test = test_df["target"]

print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)

Train shape: (400, 9)
Test shape : (100, 9)


## 6) Configurando o MLflow

Nesse exemplo vamos usar o tracking local, salvando tudo em uma pasta `mlruns`.

In [6]:
mlflow.set_tracking_uri("file://" + str(Path("mlruns").resolve()))
mlflow.set_experiment("demo-dvc-mlflow")

/Users/yandrade/PycharmProjects/ia_study/.venv/lib/python3.11/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
2026/04/07 18:57:18 INFO mlflow.tracking.fluent: Experiment with name 'demo-dvc-mlflow' does not exist. Creating a new experiment.


<Experiment: artifact_location='file:///Users/yandrade/PycharmProjects/ia_study/data/semana14_dvc_mlflow/mlruns/208610433285895489', creation_time=1775599038977, experiment_id='208610433285895489', last_update_time=1775599038977, lifecycle_stage='active', name='demo-dvc-mlflow', tags={}, trace_location=None, workspace='default'>

## 7) Treinando e logando no MLflow

In [7]:
params = {
    "C": 1.0,
    "max_iter": 200,
    "solver": "liblinear",
    "random_state": 42,
}

with mlflow.start_run(run_name="logistic_regression_baseline"):
    model = LogisticRegression(**params)
    model.fit(X_train, y_train)

    preds = model.predict(X_test)
    acc = accuracy_score(y_test, preds)

    # logs básicos
    mlflow.log_params(params)
    mlflow.log_metric("accuracy", acc)

    # salva modelo localmente
    model_path = "models/model.joblib"
    joblib.dump(model, model_path)

    # salva artefatos no MLflow
    mlflow.log_artifact(dataset_path, artifact_path="data")
    mlflow.log_artifact(model_path, artifact_path="model")

    # também registra o modelo no formato do mlflow
    mlflow.sklearn.log_model(model, artifact_path="mlflow_model")

    print(f"Accuracy: {acc:.4f}")

2026/04/07 18:57:31 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/07 18:57:31 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Accuracy: 0.8700


## 8) Relatório rápido

In [8]:
print(classification_report(y_test, preds))

              precision    recall  f1-score   support

           0       0.84      0.92      0.88        50
           1       0.91      0.82      0.86        50

    accuracy                           0.87       100
   macro avg       0.87      0.87      0.87       100
weighted avg       0.87      0.87      0.87       100



## 9) Como abrir a UI do MLflow

No terminal, rode:

```bash
mlflow ui
```

Depois abra no navegador:

```bash
http://127.0.0.1:5000
```

Lá você vai conseguir ver:
- parâmetros
- métricas
- artefatos
- execuções

## 10) Onde entra o DVC?

O DVC complementa o Git para versionar **dados** e **artefatos grandes**.

Exemplo de fluxo no terminal:

```bash
git init
dvc init

dvc add data/sample_dataset.csv
git add data/.gitignore data/sample_dataset.csv.dvc .dvc/
git commit -m "add dataset with dvc"
```

Se quiser também versionar o modelo gerado:

```bash
dvc add models/model.joblib
git add models/.gitignore models/model.joblib.dvc
git commit -m "add trained model with dvc"
```

## 11) Exemplo de remote no DVC

Você pode configurar um remote local ou em nuvem.

### Remote local
```bash
mkdir -p ../dvc-storage
dvc remote add -d localstorage ../dvc-storage
dvc push
```

### Remote S3
```bash
dvc remote add -d myremote s3://meu-bucket/dvc-storage
dvc push
```

## 12) Exemplo mínimo de pipeline com DVC

Você pode transformar treino em pipeline com um script `train.py` e um `dvc.yaml`.

### `dvc.yaml`
```yaml
stages:
  train:
    cmd: python train.py
    deps:
      - data/sample_dataset.csv
      - train.py
    outs:
      - models/model.joblib
```

Depois rodar:

```bash
dvc repro
```

Aí o DVC passa a controlar a reprodutibilidade do pipeline.

## 13) Resumo mental do combo

- **Git** → versiona código
- **DVC** → versiona dataset/modelo e pipeline
- **MLflow** → rastreia experimentos, métricas, params e artefatos

Esse trio já monta uma base bem honesta de MLOps para projetos pequenos e portfólio.

## 14) Próximos passos para evoluir esse exemplo

1. trocar dataset sintético por um CSV real  
2. criar `train.py` fora do notebook  
3. adicionar `dvc.yaml`  
4. registrar múltiplos experimentos no MLflow  
5. subir esse projeto no GitHub com um README simples